# PneumoScan --- Baseline Custom CNN

This notebook trains and evaluates a **custom 4-block CNN** as the baseline model for the PneumoScan project. The baseline serves as a control group --- a simple architecture with no pretrained weights --- against which we measure the benefit of transfer learning in subsequent notebooks.

### Why a Baseline?

- Establishes a **performance floor** for the task
- Validates the data pipeline before committing GPU hours to larger models
- Demonstrates that even a lightweight CNN (~500K parameters) can learn meaningful features from chest X-rays
- Provides a reference point for comparing transfer learning improvements

### Architecture

```
Conv2D(32) -> BN -> MaxPool
Conv2D(64) -> BN -> MaxPool
Conv2D(128) -> BN -> MaxPool
Conv2D(256) -> BN -> MaxPool
GAP -> Dense(256) -> BN -> Dropout(0.3)
     -> Dense(128) -> BN -> Dropout(0.3)
     -> Dense(3, softmax)
```

---
## 1. Environment Setup

In [ ]:
# === PneumoScan Setup ===
import os, sys, subprocess

# Detect environment and configure paths
try:
    from google.colab import drive
    drive.mount('/content/drive')
    
    REPO_DIR = '/content/PneumoScan'
    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', 'https://github.com/muhammadrakib2299/PneumoScan.git', REPO_DIR], check=True)
    else:
        subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    
    SRC_DIR = os.path.join(REPO_DIR, 'src')
    IN_COLAB = True
    print(f"Running on Google Colab | src: {SRC_DIR}")
except ImportError:
    SRC_DIR = os.path.join(os.path.dirname(os.getcwd()), 'src')
    IN_COLAB = False
    print(f"Running locally | src: {SRC_DIR}")

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# === Imports ===
import config
import data_loader
import models
import train
import evaluate
import utils

import numpy as np
import matplotlib.pyplot as plt

# Configure paths for Colab (dataset on Google Drive)
if IN_COLAB:
    config.setup_colab()

config.ensure_dirs()

print(f"Project root: {config.BASE_DIR}")
print(f"Dataset dir:  {config.RAW_DATA_DIR}")
print(f"Models dir:   {config.MODELS_DIR}")

---
## 2. Load Data

We use an 80/20 stratified train-validation split from the training directory. Data augmentation (random flip, rotation, zoom, translation, brightness) is applied only to the training split to reduce overfitting.

In [ ]:
# Load training and validation datasets with augmentation on train split
train_ds, val_ds = data_loader.load_train_val_split()

print(f"Training batches:   {len(train_ds)}")
print(f"Validation batches: {len(val_ds)}")
print(f"Batch size:         {config.BATCH_SIZE}")
print(f"Image size:         {config.IMG_SIZE}")

---
## 3. Compute Class Weights

The dataset is imbalanced (bacterial pneumonia >> viral pneumonia > normal). We compute inverse-frequency weights so the loss function penalizes errors on minority classes proportionally more.

In [ ]:
class_weights = data_loader.compute_class_weights()
print(f"\nClass weights: {class_weights}")

---
## 4. Build the Custom CNN

The baseline is a simple sequential model with 4 convolutional blocks (Conv2D + BatchNorm + MaxPool) followed by a classification head. It has roughly 500K trainable parameters --- small enough to train quickly, yet deep enough to capture meaningful spatial features.

In [ ]:
model = models.build_custom_cnn()
model.summary()

---
## 5. Train the Model

The `train.train_model()` function handles the full training pipeline:

- **For the custom CNN**, there is only **one training phase** (no frozen base to unfreeze)
- Uses Adam optimizer with learning rate 1e-3
- Callbacks: EarlyStopping (patience=7), ReduceLROnPlateau (patience=3), ModelCheckpoint
- Class weights are passed to compensate for imbalance
- Training curves are automatically saved to `outputs/figures/training_curves/`

In [ ]:
model, history_phase1, history_phase2 = train.train_model(
    "custom_cnn",
    train_ds,
    val_ds,
    class_weights=class_weights,
)

# history_phase2 will be None for custom_cnn (no fine-tuning phase)
print(f"\nPhase 2 history: {history_phase2}")

---
## 6. Training Curves

Let's visualize the training curves inline. We look for:

- **Convergence**: Are both train and val metrics improving?
- **Overfitting**: Is train accuracy much higher than val accuracy?
- **Early stopping**: Did training stop before the maximum number of epochs?

In [ ]:
# Plot training curves inline
utils.plot_training_history(history_phase1, "Custom CNN")
plt.show()

In [ ]:
# Print final training metrics
final_train_acc = history_phase1.history["accuracy"][-1]
final_val_acc = history_phase1.history["val_accuracy"][-1]
final_train_loss = history_phase1.history["loss"][-1]
final_val_loss = history_phase1.history["val_loss"][-1]

print(f"Final Training Accuracy:   {final_train_acc:.4f}")
print(f"Final Validation Accuracy: {final_val_acc:.4f}")
print(f"Final Training Loss:       {final_train_loss:.4f}")
print(f"Final Validation Loss:     {final_val_loss:.4f}")
print(f"Overfit gap (acc):         {final_train_acc - final_val_acc:.4f}")

---
## 7. Evaluate on Test Set

We evaluate on the held-out test set that the model has never seen. The evaluation pipeline computes:

- **Accuracy** --- overall correctness
- **F1 Score (macro & weighted)** --- harmonic mean of precision/recall, accounting for class imbalance
- **AUC-ROC** --- area under the ROC curve (one-vs-rest)
- **AUC-PR** --- area under the precision-recall curve
- **Cohen's Kappa** --- agreement beyond chance
- **Inference time** --- milliseconds per image

It also generates confusion matrices, ROC curves, and PR curves.

In [ ]:
# Load the test dataset
test_ds = data_loader.load_test_dataset()
print(f"Test batches: {len(test_ds)}")

In [ ]:
# Run full evaluation
metrics = evaluate.evaluate_model(model, test_ds, "custom_cnn")

print(f"\n{'=' * 40}")
print("Summary")
print(f"{'=' * 40}")
for key in ["accuracy", "f1_macro", "f1_weighted", "auc_roc", "auc_pr", "cohens_kappa", "inference_ms"]:
    print(f"  {key:18s}: {metrics[key]:.4f}")

---
## Summary

| Aspect | Details |
|--------|--------|
| Model | Custom 4-block CNN (~500K params) |
| Training | Single phase, 10 epochs max, Adam (lr=1e-3) |
| Augmentation | Random flip, rotation, zoom, translation, brightness |
| Class weights | Applied to handle imbalanced classes |
| Purpose | Establishes the performance baseline |

### Key Observations

- The custom CNN provides a reasonable starting point but is limited by its shallow architecture and lack of pretrained features.
- Any accuracy gap between train and val suggests the model could benefit from regularization or more data --- both of which transfer learning addresses.
- These baseline metrics will be directly compared against ResNet-50, EfficientNet-B0, DenseNet-121, and MobileNetV2 in later notebooks.

**Next:** Proceed to `03_resnet50_transfer.ipynb` to train ResNet-50 with transfer learning and compare against this baseline.